In [44]:
os.chdir("/Users/tanvigambhir/Documents/SQL/neuro_analysis_data")

In [45]:
import pandas as pd

patients_df = pd.read_csv('PATIENTS.csv')
patients_df.head()
patients_df.info()
patients_df_clean = patients_df.drop(columns=['row_id'])
patients_df_clean.to_sql('patients', conn, if_exists='append', index=False)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM patients")
print(cursor.fetchone())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   row_id       100 non-null    int64 
 1   subject_id   100 non-null    int64 
 2   gender       100 non-null    object
 3   dob          100 non-null    object
 4   dod          100 non-null    object
 5   dod_hosp     70 non-null     object
 6   dod_ssn      77 non-null     object
 7   expire_flag  100 non-null    int64 
dtypes: int64(3), object(5)
memory usage: 6.4+ KB


IntegrityError: UNIQUE constraint failed: patients.subject_id

In [ ]:
admissions_df = pd.read_csv('ADMISSIONS.csv')
admissions_df.info
admissions_df_clean = admissions_df.drop(columns=['row_id'])
admissions_df_clean.to_sql('admissions', conn, if_exists='append', index=False)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM admissions")
print(cursor.fetchone())

In [57]:
d_labitems_df = pd.read_csv('D_LABITEMS.csv')
d_labitems_df.info()
d_labitems_df.head()

diagnoses_df = pd.read_csv('DIAGNOSES_ICD.csv')
diagnoses_df.info()

d_icd_df = pd.read_csv('D_ICD_DIAGNOSES.csv')
d_icd_df.info()

icustays_df_clean = icustays_df.drop(columns=['row_id'])
icustays_df_clean.to_sql('icustays', conn, if_exists='append', index=False)
conn.commit()
cursor.execute("SELECT COUNT(*) FROM icustays")
print(cursor.fetchone())

d_icd_df_clean = d_icd_df.drop(columns=['row_id'])
d_icd_df_clean.to_sql('d_icd_diagnoses', conn, if_exists='append', index=False)
conn.commit()
cursor.execute("SELECT COUNT(*) FROM d_icd_diagnoses")
print(cursor.fetchone())

diagnoses_df_clean = diagnoses_df.drop(columns=['row_id'])
diagnoses_df_clean.to_sql('diagnoses_icd', conn, if_exists='append', index=False)
conn.commit()
cursor.execute("SELECT COUNT(*) FROM diagnoses_icd")
print(cursor.fetchone())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 753 entries, 0 to 752
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   row_id      753 non-null    int64 
 1   itemid      753 non-null    int64 
 2   label       753 non-null    object
 3   fluid       753 non-null    object
 4   category    753 non-null    object
 5   loinc_code  585 non-null    object
dtypes: int64(2), object(4)
memory usage: 35.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1761 entries, 0 to 1760
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   row_id      1761 non-null   int64 
 1   subject_id  1761 non-null   int64 
 2   hadm_id     1761 non-null   int64 
 3   seq_num     1761 non-null   int64 
 4   icd9_code   1761 non-null   object
dtypes: int64(4), object(1)
memory usage: 68.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14567 entries, 0 to 14566
Data columns

IntegrityError: UNIQUE constraint failed: icustays.icustay_id

In [ ]:
neuro_keywords = 'parkinson|alzheimer|dementia|epilepsy|seizure|stroke|multiple sclerosis'

neuro_codes = d_icd_df[d_icd_df['long_title'].str.contains(neuro_keywords, case=False, na=False)]
print(neuro_codes.shape)
neuro_codes[['icd9_code', 'long_title']].head(20)

In [ ]:
neuro_codes_used = diagnoses_df[diagnoses_df['icd9_code'].isin(neuro_codes['icd9_code'])]
print(neuro_codes_used.shape)
neuro_codes_used.merge(d_icd_df[['icd9_code', 'long_title']], on='icd9_code')[['subject_id', 'hadm_id', 'icd9_code', 'long_title']]

In [ ]:
icustays_df = pd.read_csv('ICUSTAYS.csv')
icustays_df.info()

prescriptions_df = pd.read_csv('PRESCRIPTIONS.csv')
prescriptions_df.info()

labevents_df = pd.read_csv('LABEVENTS.csv')
labevents_df.info()

prescriptions_df_clean = prescriptions_df.drop(columns=['row_id'])
prescriptions_df_clean.to_sql('prescriptions', conn, if_exists='append', index=False)
conn.commit()
cursor.execute("SELECT COUNT(*) FROM prescriptions")
print(cursor.fetchone())

labevents_df_clean = labevents_df.drop(columns=['row_id'])
labevents_df_clean.to_sql('labevents', conn, if_exists='append', index=False)
conn.commit()
cursor.execute("SELECT COUNT(*) FROM labevents")
print(cursor.fetchone())

In [ ]:
import sqlite3

conn = sqlite3.connect('mimic_neuro.db')
cursor = conn.cursor()

In [ ]:
cursor.execute("DROP TABLE IF EXISTS patients")
cursor.execute('''
CREATE TABLE patients (
    subject_id INTEGER PRIMARY KEY,
    gender TEXT,
    dob TEXT,
    dod TEXT,
    dod_hosp TEXT,
    dod_ssn TEXT,
    expire_flag INTEGER
)
''')

In [ ]:
cursor.execute("DROP TABLE IF EXISTS admissions")
cursor.execute('''
CREATE TABLE admissions(
    hadm_id INTEGER PRIMARY KEY,
    subject_id INTEGER,
    admittime TEXT, 
    dischtime TEXT, 
    deathtime TEXT, 
    edregtime TEXT, 
    edouttime TEXT,
    admission_type TEXT,
    ethnicity TEXT,
    admission_location TEXT, 
    discharge_location TEXT, 
    insurance TEXT, 
    language TEXT, 
    religion TEXT, 
    marital_status TEXT, 
    diagnosis TEXT,
    hospital_expire_flag INTEGER,
    has_chartevents_data INTEGER,
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id)
)
''')

In [ ]:
cursor.execute("DROP TABLE IF EXISTS icustays")
cursor.execute('''
CREATE TABLE icustays(
    icustay_id INTEGER PRIMARY KEY,
    subject_id INTEGER,
    hadm_id INTEGER,
    dbsource TEXT,
    first_careunit TEXT,
    last_careunit TEXT,
    first_wardid INTEGER,
    last_wardid INTEGER,
    intime TEXT,
    outtime TEXT,
    los REAL,
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id),
    FOREIGN KEY (hadm_id) REFERENCES admissions(hadm_id)
)
''')

In [ ]:
cursor.execute("DROP TABLE IF EXISTS d_icd_diagnoses")
cursor.execute('''
CREATE TABLE d_icd_diagnoses(
    icd9_code TEXT PRIMARY KEY,
    short_title TEXT,
    long_title TEXT
)
''')

In [ ]:
cursor.execute("DROP TABLE IF EXISTS diagnoses_icd")
cursor.execute('''
CREATE TABLE diagnoses_icd (
    hadm_id     INTEGER,
    subject_id  INTEGER,
    seq_num     INTEGER,
    icd9_code   TEXT,
    PRIMARY KEY (hadm_id, icd9_code),
    FOREIGN KEY (hadm_id) REFERENCES admissions(hadm_id),
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id),
    FOREIGN KEY (icd9_code) REFERENCES d_icd_diagnoses(icd9_code)
)
''')

In [ ]:
cursor.execute("DROP TABLE IF EXISTS prescriptions")
cursor.execute('''
CREATE TABLE prescriptions (
    row_id INTEGER PRIMARY KEY,
    subject_id INTEGER,
    hadm_id INTEGER,
    icustay_id INTEGER,
    startdate TEXT,
    enddate TEXT,
    drug_type TEXT,
    drug TEXT,
    drug_name_poe TEXT,
    drug_name_generic TEXT,
    formulary_drug_cd TEXT,
    gsn REAL,
    ndc REAL,
    prod_strength TEXT,
    dose_val_rx TEXT,
    dose_unit_rx TEXT,
    form_val_disp TEXT,
    form_unit_disp TEXT,
    route TEXT,
    FOREIGN KEY (hadm_id) REFERENCES admissions(hadm_id),
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id),
    FOREIGN KEY (icustay_id) REFERENCES icustays(icustay_id)
)
''')

In [ ]:
cursor.execute("DROP TABLE IF EXISTS labevents")
cursor.execute('''
CREATE TABLE labevents (
    row_id INTEGER PRIMARY KEY,
    subject_id INTEGER,
    hadm_id INTEGER,
    itemid INTEGER,
    charttime TEXT,
    value TEXT,
    valueuom TEXT,
    valuenum REAL,
    flag TEXT,
    FOREIGN KEY (subject_id) REFERENCES patients(subject_id),
    FOREIGN KEY (hadm_id) REFERENCES admissions(hadm_id)
)
''')

In [59]:
cursor.execute("DROP TABLE IF EXISTS d_labitems")
cursor.execute('''
CREATE TABLE d_labitems (
    itemid INTEGER PRIMARY KEY,
    label TEXT,
    fluid TEXT,
    category TEXT,
    loinc_code TEXT
)
''')
d_labitems_df_clean = d_labitems_df.drop(columns=['row_id'])
d_labitems_df_clean.to_sql('d_labitems', conn, if_exists='append', index=False)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM d_labitems")
print(cursor.fetchone())

(753,)


In [49]:
cursor.execute('''
WITH neuro_diagnoses AS (
    SELECT hadm_id, subject_id, icd9_code
    FROM diagnoses_icd
    WHERE icd9_code IN (
        SELECT icd9_code
        FROM d_icd_diagnoses
        WHERE long_title LIKE '%parkinson%' OR long_title LIKE '%alzheimer%' OR long_title LIKE '%dementia%' OR long_title LIKE '%epilepsy%' OR long_title LIKE '%stroke%' OR long_title LIKE '%multiple sclerosis%' OR long_title LIKE '%seizure%'  
    )
)
SELECT * FROM neuro_diagnoses
''')
print(cursor.fetchall())

[(189483, 10036, '3310'), (189483, 10036, '29410'), (146672, 10069, '34591'), (149044, 10088, '29041'), (168233, 10088, '29040'), (104697, 40612, '34590'), (171628, 42066, '34590'), (167754, 42302, '340'), (114648, 42321, '3310'), (114648, 42321, '29410'), (139932, 42367, '34570')]


In [50]:
cursor.execute('''
WITH neuro_admissions AS (
    SELECT DISTINCT hadm_id, subject_id
    FROM diagnoses_icd
    WHERE icd9_code IN (
        SELECT icd9_code
        FROM d_icd_diagnoses
        WHERE long_title LIKE '%parkinson%' OR long_title LIKE '%alzheimer%' OR long_title LIKE '%dementia%' OR long_title LIKE '%epilepsy%' OR long_title LIKE '%stroke%' OR long_title LIKE '%multiple sclerosis%' OR long_title LIKE '%seizure%'
    )
)
SELECT na.hadm_id, na.subject_id, icu.los
FROM neuro_admissions na
JOIN icustays icu ON na.hadm_id = icu.hadm_id
''')
print(cursor.fetchall())

[(114648, 42321, 1.9121), (189483, 10036, 1.8068), (104697, 40612, 3.5465), (171628, 42066, 5.0956), (168233, 10088, 1.6226), (139932, 42367, 29.2601), (167754, 42302, 0.9911), (149044, 10088, 2.0091), (146672, 10069, 15.041)]


In [51]:
cursor.execute('''
WITH neuro_admissions AS (
    SELECT DISTINCT hadm_id, subject_id
    FROM diagnoses_icd
    WHERE icd9_code IN (
        SELECT icd9_code
        FROM d_icd_diagnoses
        WHERE long_title LIKE '%parkinson%' OR long_title LIKE '%alzheimer%' OR long_title LIKE '%dementia%' OR long_title LIKE '%epilepsy%' OR long_title LIKE '%stroke%' OR long_title LIKE '%multiple sclerosis%' OR long_title LIKE '%seizure%'
    )
)
SELECT 
    na.hadm_id, 
    na.subject_id, 
    icu.los,
    RANK() OVER (ORDER BY icu.los DESC) AS los_rank
FROM neuro_admissions na
JOIN icustays icu ON na.hadm_id = icu.hadm_id
''')
print(cursor.fetchall())

[(139932, 42367, 29.2601, 1), (146672, 10069, 15.041, 2), (171628, 42066, 5.0956, 3), (104697, 40612, 3.5465, 4), (149044, 10088, 2.0091, 5), (114648, 42321, 1.9121, 6), (189483, 10036, 1.8068, 7), (168233, 10088, 1.6226, 8), (167754, 42302, 0.9911, 9)]


In [52]:
cursor.execute('''
WITH neuro_admissions AS (
    SELECT DISTINCT hadm_id, subject_id
    FROM diagnoses_icd
    WHERE icd9_code IN (
        SELECT icd9_code
        FROM d_icd_diagnoses
        WHERE long_title LIKE '%parkinson%' OR long_title LIKE '%alzheimer%' OR long_title LIKE '%dementia%' OR long_title LIKE '%epilepsy%' OR long_title LIKE '%stroke%' OR long_title LIKE '%multiple sclerosis%' OR long_title LIKE '%seizure%'
    )
)
SELECT
    (SELECT AVG(los) FROM icustays WHERE hadm_id IN (SELECT hadm_id FROM neuro_admissions)) AS neuro_avg_los,
    (SELECT AVG(los) FROM icustays) AS overall_avg_los
''')
print(cursor.fetchall())

[(6.809433333333334, 4.452456617647059)]


In [53]:
cursor.execute('''
WITH neuro_admissions AS (
    SELECT DISTINCT hadm_id, subject_id
    FROM diagnoses_icd
    WHERE icd9_code IN (
        SELECT icd9_code
        FROM d_icd_diagnoses
        WHERE long_title LIKE '%parkinson%' OR long_title LIKE '%alzheimer%' OR long_title LIKE '%dementia%' OR long_title LIKE '%epilepsy%' OR long_title LIKE '%stroke%' OR long_title LIKE '%multiple sclerosis%' OR long_title LIKE '%seizure%'
    )
)
SELECT
    (SELECT AVG(hospital_expire_flag) FROM admissions WHERE hadm_id IN (SELECT hadm_id FROM neuro_admissions)) AS neuro_mortality_rate,
    (SELECT AVG(hospital_expire_flag) FROM admissions) AS overall_mortality_rate
''')
print(cursor.fetchall())

[(0.4444444444444444, 0.31007751937984496)]


In [54]:
cursor.execute('''
WITH neuro_admissions AS (
    SELECT DISTINCT hadm_id, subject_id
    FROM diagnoses_icd
    WHERE icd9_code IN (
        SELECT icd9_code
        FROM d_icd_diagnoses
        WHERE long_title LIKE '%parkinson%' OR long_title LIKE '%alzheimer%' OR long_title LIKE '%dementia%' OR long_title LIKE '%epilepsy%' OR long_title LIKE '%stroke%' OR long_title LIKE '%multiple sclerosis%' OR long_title LIKE '%seizure%'
    )
)
SELECT drug, COUNT(*) AS times_prescribed
FROM prescriptions
WHERE hadm_id IN (SELECT hadm_id FROM neuro_admissions)
GROUP BY drug
ORDER BY times_prescribed DESC
''')
print(cursor.fetchall())

[('Potassium Chloride', 55), ('D5W', 37), ('NS', 29), ('SW', 28), ('Insulin', 26), ('Furosemide', 25), ('0.9% Sodium Chloride', 20), ('Metoprolol Tartrate', 19), ('Metoprolol', 19), ('Morphine Sulfate', 18), ('Magnesium Sulfate', 15), ('Sodium Chloride 0.9%  Flush', 14), ('Iso-Osmotic Dextrose', 12), ('Calcium Gluconate', 12), ('Acetaminophen', 12), ('5% Dextrose', 12), ('Heparin', 11), ('Potassium Phosphate', 10), ('Vancomycin', 9), ('PHENObarbital', 9), ('Lorazepam', 9), ('LeVETiracetam', 9), ('Fosphenytoin', 9), ('PredniSONE', 8), ('Bisacodyl', 8), ('LR', 7), ('Docusate Sodium', 7), ('Vancomycin HCl', 6), ('Soln', 6), ('PENTObarbital Sodium', 6), ('Midazolam', 6), ('HydrALAzine', 6), ('Albuterol 0.083% Neb Soln', 6), ('Scopolamine Patch', 5), ('Propofol', 5), ('Phenytoin Sodium', 5), ('Lisinopril', 5), ('Haloperidol', 5), ('Fentanyl Citrate', 5), ('Levofloxacin', 4), ('HYDROmorphone (Dilaudid)', 4), ('Docusate Sodium (Liquid)', 4), ('CefePIME', 4), ('Atorvastatin', 4), ('Aspirin', 4

In [61]:
cursor.execute('''
WITH neuro_admissions AS (
    SELECT DISTINCT hadm_id, subject_id
    FROM diagnoses_icd
    WHERE icd9_code IN (
        SELECT icd9_code
        FROM d_icd_diagnoses
        WHERE long_title LIKE '%parkinson%' OR long_title LIKE '%alzheimer%' OR long_title LIKE '%dementia%' OR long_title LIKE '%epilepsy%' OR long_title LIKE '%stroke%' OR long_title LIKE '%multiple sclerosis%' OR long_title LIKE '%seizure%'
    )
)
SELECT d.label, COUNT(*) AS times_flagged
FROM labevents le
JOIN d_labitems d ON le.itemid = d.itemid
WHERE le.hadm_id IN (SELECT hadm_id FROM neuro_admissions)
AND le.flag = 'abnormal'
GROUP BY d.label
ORDER BY times_flagged DESC
''')
print(cursor.fetchall())

[('Hematocrit', 99), ('Hemoglobin', 92), ('Red Blood Cells', 88), ('Glucose', 86), ('Urea Nitrogen', 57), ('Calcium, Total', 52), ('Chloride', 39), ('RDW', 37), ('PT', 31), ('White Blood Cells', 24), ('INR(PT)', 22), ('Phosphate', 21), ('MCH', 20), ('Phenytoin', 18), ('MCHC', 17), ('Creatinine', 17), ('pO2', 15), ('Sodium', 15), ('Potassium', 15), ('Bicarbonate', 15), ('PTT', 13), ('pH', 12), ('Albumin', 12), ('Lymphocytes', 11), ('Neutrophils', 10), ('Troponin T', 9), ('Osmolality, Measured', 9), ('Magnesium', 9), ('pCO2', 8), ('Vancomycin', 8), ('Valproic Acid', 8), ('Free Calcium', 8), ('Calculated Total CO2', 8), ('Cortisol', 7), ('Ammonia', 7), ('Potassium, Whole Blood', 6), ('Platelet Count', 6), ('Phenytoin, Percent Free', 6), ('WBC', 5), ('MCV', 5), ('Creatine Kinase, MB Isoenzyme', 5), ('CK-MB Index', 5), ('Vitamin B12', 4), ('Thyroxine (T4)', 4), ('Thyroid Stimulating Hormone', 4), ('RBC', 4), ('Phenobarbital', 4), ('Triiodothyronine (T3)', 3), ('Thyroxine (T4), Free', 3), ('